In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# AVL Trees

This notebook implements
[AVL trees](https://en.wikipedia.org/wiki/AVL_tree).  The set $\mathcal{A}$ of *AVL trees* is defined inductively:

  - $\texttt{Nil} \in \mathcal{A}$.
  - $\texttt{Node}(k,v,l,r) \in \mathcal{A}\quad$  if 
      - $\texttt{Node}(k,v,l,r) \in \mathcal{B}$,
      - $l, r \in \mathcal{A}$, and
      - $|l.\texttt{height}() - r.\texttt{height}()| \leq 1$.

According to this definition, an AVL tree is an *ordered binary tree*
such that for every node $\texttt{Node}(k,v,l,r)$ in this tree the height of the left subtree $l$ and the right
subtree  $r$ differ at most by one. 

The class `AVLTree` represents the nodes of an AVL tree.  This class has the following member variables:

  - `mKey` is the key stored at the root of the tree,
  - `mValue` is the values associated with this key,
  - `mLeft` is the left subtree, 
  - `mRight` is the right subtree, and
  - `mHeight` is the height.

The constructor creates the empty tree. 

We also define an interface for the `AVLTree` class so TypeScript allows us to reference methods that will be implemented later.

In [ ]:
class AVLTree<K extends string | number, V> {
    static sNodeCount: number = 0;
    mKey:    K             | null;
    mValue:  V             | null;
    mLeft:   AVLTree<K, V> | null;
    mRight:  AVLTree<K, V> | null;
    mHeight: number;
    mID:     number | null;

    constructor(
        key:   K             | null = null, 
        value: V             | null = null, 
        left:  AVLTree<K, V> | null = null, 
        right: AVLTree<K, V> | null = null
    ) {
        if (key === null || left === null || right === null) {
            this.mKey    = null;
            this.mValue  = null;
            this.mLeft   = null;
            this.mRight  = null;
            this.mHeight = 0;
        } else {
            this.mKey    = key;
            this.mValue  = value;
            this.mLeft   = left;
            this.mRight  = right;
            this.mHeight = Math.max(left.mHeight, right.mHeight) + 1;
        }
        this.mID = null;
    }
}

interface AVLTree<K extends string | number, V> {
    isEmpty(): boolean;
    isNode(): this is AVLTree<K, V> & { mKey: K, mValue: V, mLeft: AVLTree<K, V>, mRight: AVLTree<K, V> };
    find(key: K): V | null | undefined;
    insert(key: K, value: V | null): AVLTree<K, V>;
    delete(key: K): AVLTree<K, V>;
    _delMin(): [AVLTree<K, V>, K, V | null];
    _restore(): AVLTree<K, V>;
    toDot(): Promise<string>;
    _assignIDs(NodeDict: Map<number, AVLTree<K, V>>): void;
    maxKey(): K | null;
}

Given an ordered binary tree $t$, the expression $t.\texttt{isEmpty}()$ checks whether $t$ is the empty tree. The `isNode()` function serves as a modern TypeScript Type Guard to eliminate null checks.

In [ ]:
AVLTree.prototype.isEmpty = function<K extends string | number, V>(this: AVLTree<K, V>): boolean {
    return this.mKey === null;
};

AVLTree.prototype.isNode = function<K extends string | number, V>(this: AVLTree<K, V>): boolean {
    return this.mKey !== null && this.mLeft !== null && this.mRight !== null;
} as any;

Given an ordered binary tree $t$ and a key $k$, the expression $t.\texttt{find}(k)$ returns the value stored unter the key $k$.
The method `find` is defined inductively as follows:
  - $\texttt{Nil}.\texttt{find}(k) = \Omega$,

    because the empty tree is interpreted as the empty map.
  - $\texttt{Node}(k, v, l, r).\texttt{find}(k) = v$,
      
    because the node $\texttt{Node}(k,v,l,r)$ stores the assignment $k \mapsto v$.
  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = l.\texttt{find}(k_1)$,

    because if $k_1$ is less than $k_2$, then any mapping for $k_1$ has to be stored in the left subtree  $l$.
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = r.\texttt{find}(k_1)$,

    because if $k_1$ is greater than $k_2$, then any mapping for $k_1$ has to be stored in the right subtree  $r$.

In [ ]:
AVLTree.prototype.find = function<K extends string | number, V>(this: AVLTree<K, V>, key: K): V | null | undefined {
    if (!this.isNode())   return undefined;
    if (this.mKey == key) return this.mValue;
    if (key < this.mKey)  return this.mLeft.find(key);
    return this.mRight.find(key);
};


The method  $\texttt{insert}()$ is specified via recursive equations.
  - $\texttt{Nil}.\texttt{insert}(k,v) = \texttt{Node}(k,v, \texttt{Nil}, \texttt{Nil})$,
  - $\texttt{Node}(k, v_2, l, r).\texttt{insert}(k,v_1) = \texttt{Node}(k, v_1, l, r)$,
  - $k_1 < k_2 \rightarrow 
          \texttt{Node}(k_2, v_2, l, r).\texttt{insert}(k_1, v_1) =
          \texttt{Node}\bigl(k_2, v_2, l.\texttt{insert}(k_1,v_1), r\bigr).\texttt{restore}()$,
  - $k_1 > k_2 \rightarrow 
         \texttt{Node}(k_2, v_2, l, r).\texttt{insert}\bigl(k_1, v_1\bigr) = 
         \texttt{Node}\bigl(k_2, v_2, l, r.\texttt{insert}(k_1,v_1)\bigr).\texttt{restore}()$.

The function $\texttt{restore}$ is an auxiliary function that is defined below.  This function restores the balancing condition if it is violated after an insertion.

In [ ]:
AVLTree.prototype.insert = function<K extends string | number, V>(this: AVLTree<K, V>, key: K, value: V | null): AVLTree<K, V> {
    if (!this.isNode()) {
        return new AVLTree(key, value, new AVLTree<K, V>(), new AVLTree<K, V>());
    }
    if (key == this.mKey) {
        return new AVLTree(key, value, this.mLeft, this.mRight);
    }
    if (key < this.mKey) {
        return new AVLTree(this.mKey, this.mValue, this.mLeft.insert(key, value), this.mRight)._restore();
    }
    return new AVLTree(this.mKey, this.mValue, this.mLeft, this.mRight.insert(key, value))._restore();
};

The method $\texttt{self}.\texttt{delete}(k)$ removes the key $k$ from the tree $\texttt{self}$.  It is defined as follows:

  - $\texttt{Nil}.\texttt{delete}(k) = \texttt{Nil}$,
  - $\texttt{Node}(k,v,\texttt{Nil},r).\texttt{delete}(k) = r$,
  - $\texttt{Node}(k,v,l,\texttt{Nil}).\texttt{delete}(k) = l$,
  - $l \not= \texttt{Nil} \,\wedge\, r \not= \texttt{Nil} \,\wedge\, 
       \langle r',k_{min}, v_{min} \rangle := r.\texttt{delMin}()  \;\rightarrow\;
      \texttt{Node}(k,v,l,r).\texttt{delete}(k) = \texttt{Node}(k_{min},v_{min},l,r').\texttt{restore}()$
  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}(k_1) = 
    \texttt{Node}\bigl(k_2,v_2,l.\texttt{delete}(k_1),r\bigr).\texttt{restore}()$,
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}(k_1) = 
     \texttt{Node}\bigl(k_2,v_2,l,r.\texttt{delete}(k_1)\bigr).\texttt{restore}()$.

In [ ]:
AVLTree.prototype.delete = function<K extends string | number, V>(this: AVLTree<K, V>, key: K): AVLTree<K, V> {
    if (!this.isNode()) return this;
    if (key == this.mKey) {
        if (this.mLeft .isEmpty()) return this.mRight;
        if (this.mRight.isEmpty()) return this.mLeft;
        const [newRight, minKey, minValue] = this.mRight._delMin();
        return new AVLTree(minKey, minValue, this.mLeft, newRight)._restore();
    }
    if (key < this.mKey) {
        return new AVLTree(this.mKey, this.mValue, this.mLeft.delete(key), this.mRight)._restore();
    }
    return new AVLTree(this.mKey, this.mValue, this.mLeft, this.mRight.delete(key))._restore();
};

The method $\texttt{self}.\texttt{delMin}()$ removes the smallest key from the given tree $\texttt{self}$
and returns a triple of the form
$$(\texttt{self}, k_m, v_m)$$
where $\texttt{self}$ is the tree that remains after removing the smallest key, while $k_m$ is the smallest key that has been found and $v_m$ is the associated value.  

The function is defined as follows:

  - $\texttt{Node}(k, v, \texttt{Nil}, r).\texttt{delMin}() = \langle r, k, v \rangle$,
  - $l\not= \texttt{Nil} \wedge \langle l',k_{min}, v_{min}\rangle := l.\texttt{delMin}() 
       \;\rightarrow\;
     \texttt{Node}(k, v, l, r).\texttt{delMin}() = 
     \langle \texttt{Node}(k, v, l', r).\texttt{restore}(), k_{min}, v_{min} \rangle
    $

In [ ]:
AVLTree.prototype._delMin = function<K extends string | number, V>(this: AVLTree<K, V>): [AVLTree<K, V>, K, V | null] {
    if (!this.isNode()) throw new Error("Called _delMin on empty tree");
    if (this.mLeft.isEmpty()) {
        return [this.mRight, this.mKey, this.mValue];
    }
    const [newLeft, minKey, minValue] = this.mLeft._delMin();
    const restoredTree = new AVLTree(this.mKey, this.mValue, newLeft, this.mRight)._restore();
    return [restoredTree, minKey, minValue];
};

The function $\texttt{restore}(\texttt{self})$ restores the balancing condition of the given binary tree
at the root node and recompute the variable $\texttt{mHeight}$.

The method $\texttt{restore}$ is specified via conditional equations.

  - $\texttt{Nil}.\texttt{restore}() = \texttt{Nil}$,

    because the empty tree already is an  AVL tree.
  - $|l.\texttt{height}() - r.\texttt{height}()| \leq 1 \rightarrow 
       \texttt{Node}(k,v,l,r).\texttt{restore}() = \texttt{Node}(k,v,l,r)$.

      If the balancing condition is satisfied, then nothing needs to be done. 
  - $\begin{array}[t]{cl}
              & l_1.\texttt{height}() = r_1.\texttt{height}() + 2    \\ 
       \wedge & l_1 = \texttt{Node}(k_2,v_2,l_2,r_2)                 \\
       \wedge & l_2.\texttt{height}() \geq r_2.\texttt{height}()     \\[0.2cm]
       \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                     \texttt{Node}\bigl(k_2,v_2,l_2,\texttt{Node}(k_1,v_1,r_2,r_1)\bigr)
       \end{array}
    $
  - $\begin{array}[t]{cl}
               & l_1.\texttt{height}() = r_1.\texttt{height}() + 2    \\ 
        \wedge & l_1 = \texttt{Node}(k_2,v_2,l_2,r_2)               \\
        \wedge & l_2.\texttt{height}() < r_2.\texttt{height}()     \\
        \wedge & r_2 = \texttt{Node}(k_3,v_3,l_3,r_3)               \\
        \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                      \texttt{Node}\bigl(k_3,v_3,\texttt{Node}(k_2,v_2,l_2,l_3),\texttt{Node}(k_1,v_1,r_3,r_1) \bigr)
        \end{array}
    $

  - $\begin{array}[t]{cl}
              & r_1.\texttt{height}() = l_1.\texttt{height}() + 2    \\ 
       \wedge & r_1 = \texttt{Node}(k_2,v_2,l_2,r_2)               \\
       \wedge & r_2.\texttt{height}() \geq l_2.\texttt{height}()     \\[0.2cm]
       \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                     \texttt{Node}\bigl(k_2,v_2,\texttt{Node}(k_1,v_1,l_1,l_2),r_2\bigr)
       \end{array}
    $
  - $\begin{array}[t]{cl}
               & r_1.\texttt{height}() = l_1.\texttt{height}() + 2    \\ 
        \wedge & r_1 = \texttt{Node}(k_2,v_2,l_2,r_2)               \\
        \wedge & r_2.\texttt{height}() < l_2.\texttt{height}()     \\
        \wedge & l_2 = \texttt{Node}(k_3,v_3,l_3,r_3)               \\
        \rightarrow & \texttt{Node}(k_1,v_1,l_1,r_1).\texttt{restore}() = 
                      \texttt{Node}\bigl(k_3,v_3,\texttt{Node}(k_1,v_1,l_1,l_3),\texttt{Node}(k_2,v_2,r_3,r_2) \bigr)
        \end{array}
    $

In [ ]:
AVLTree.prototype._restore = function<K extends string | number, V>(this: AVLTree<K, V>): AVLTree<K, V> {
    if (!this.isNode()) return this;
    
    const bal = this.mLeft.mHeight - this.mRight.mHeight;
    if (Math.abs(bal) <= 1) return this;

    const k1 = this.mKey, v1 = this.mValue;

    if (bal === 2 && this.mLeft.isNode()) {
        const l1 = this.mLeft, r1 = this.mRight;
        const k2 = l1.mKey, v2 = l1.mValue, l2 = l1.mLeft, r2 = l1.mRight;

        if (l2.mHeight >= r2.mHeight) {
            return new AVLTree(k2, v2, l2, new AVLTree(k1, v1, r2, r1));
        }
        if (r2.isNode()) {
            return new AVLTree(r2.mKey, r2.mValue, new AVLTree(k2, v2, l2, r2.mLeft), new AVLTree(k1, v1, r2.mRight, r1));
        }
    }

    if (bal === -2 && this.mRight.isNode()) {
        const l1 = this.mLeft, r1 = this.mRight;
        const k2 = r1.mKey, v2 = r1.mValue, l2 = r1.mLeft, r2 = r1.mRight;

        if (r2.mHeight >= l2.mHeight) {
            return new AVLTree(k2, v2, new AVLTree(k1, v1, l1, l2), r2);
        }
        if (l2.isNode()) {
            return new AVLTree(l2.mKey, l2.mValue, new AVLTree(k1, v1, l1, l2.mLeft), new AVLTree(k2, v2, l2.mRight, r2));
        }
    }
    return this;
};

Given an ordered binary tree, this function renders the tree graphically using `Graphviz`.

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";

AVLTree.prototype.toDot = async function<K extends string | number, V>(this: AVLTree<K, V>): Promise<string> {
    AVLTree.sNodeCount = 0;
    
    let dot = `digraph G {\n        node [shape=record style=rounded];\n`;
    
    const nodes = new Map<number, AVLTree<K, V>>();
    this._assignIDs(nodes);

    for (const [n, t] of nodes.entries()) {
        if (t.mValue != null && t.mKey != null) {
            dot += `        ${n} [label="{${t.mKey}|${t.mValue}}"];\n`;
        } else if (t.mKey != null) {
            dot += `        ${n} [label="${t.mKey}"];\n`;
        } else {
            dot += `        ${n} [label="" shape=point];\n`;
        }
    }

    for (const [n, t] of nodes.entries()) {
        if (t.isNode()) {
            dot += `        ${n} -> ${t.mLeft.mID};\n`;
            dot += `        ${n} -> ${t.mRight.mID};\n`;
        }
    }

    dot += `    }`;
    return dot;
};

This method assigns a unique identifier with each node.  The dictionary `NodeDict` maps these identifiers to the nodes where they occur.

In [ ]:
AVLTree.prototype._assignIDs = function<K extends string | number, V>(this: AVLTree<K, V>, NodeDict: Map<number, AVLTree<K, V>>): void {
    AVLTree.sNodeCount += 1;
    this.mID = AVLTree.sNodeCount;
    NodeDict.set(this.mID, this);
    
    if (this.isNode()) {
        this.mLeft._assignIDs(NodeDict);
        this.mRight._assignIDs(NodeDict);
    }
};

The function $\texttt{demo}()$ creates a small ordered binary tree.

In [ ]:
function demo(): AVLTree<string, number> {
    let m = new AVLTree<string, number>();
    m = m.insert("anton", 123);
    m = m.insert("hugo", 345);
    m = m.insert("gustav", 789);
    m = m.insert("jens", 234);
    m = m.insert("hubert", 432);
    m = m.insert("andre", 342);
    m = m.insert("philip", 342);
    m = m.insert("rene", 345);
    return m;
}

The function `renderTree` is used to render a AVLTree to the screen.

In [ ]:
import { display } from "tslab";

async function renderTree<K extends string | number, V>(t: AVLTree<K, V>): Promise<void> {
    const dot = await t.toDot();
    const gv = await Graphviz.load();
    const svg = await gv.layout(dot, "svg", "dot");
    display.html(svg);
}

In [ ]:
let t = demo();
await renderTree(t);

In [ ]:
t = t.delete('gustav');
await renderTree(t);

In [ ]:
t = t.delete('hubert');
await renderTree(t);

Let's generate an ordered binary tree with random keys.

In [ ]:
let t2 = new AVLTree<number, null>();
for (let i = 0; i < 30; i++) {
    const k = Math.floor(Math.random() * 100);
    t2 = t2.insert(k, null);
}
await renderTree(t2);

This tree looks more or less balanced.  Lets us try to create a tree by inserting sorted numbers because that resulted in linear complexity for ordered binary trees.

In [ ]:
let t3 = new AVLTree<number, null>();
for (let k = 0; k < 30; k++) {
    t3 = t3.insert(k, null);
}
await renderTree(t3);

Next, we compute the set of prime numbers $\leq 100$.  Mathematically, this set is given as follows:
$$ \bigl\{2, \cdots, 100 \bigr\} - \bigl\{ i \cdot j \bigm| i, j \in \{2, \cdots, 100 \}\bigr\}$$

In [ ]:
let S = new AVLTree<number, null>();
for (let k = 2; k <= 100; ++k) {
    S = S.insert(k, null);
}
for (let i = 2; i <= 100; ++i) {
    for (let j = 2; j <= 100; ++j) {
        S = S.delete(i * j);
    }
}
await renderTree(S);

The function $t.\texttt{maxKey}()$ returns the biggest key in the tree $t$.  It is defined inductively:
  - $\texttt{Nil}.\texttt{maxKey}() = \Omega$,
  - $\texttt{Node}(k,v,l,\texttt{Nil}).\texttt{maxKey}() = k$,
  - $r \not= \texttt{Nil} \rightarrow \texttt{Node}(k,v,l,r).\texttt{maxKey}() = r.\texttt{maxKey}()$.

In [ ]:
AVLTree.prototype.maxKey = function<K extends string | number, V>(this: AVLTree<K, V>): K | null {
    if (!this.isNode()) return null;
    if (this.mRight.isEmpty()) return this.mKey;
    return this.mRight.maxKey();
};

The function $\texttt{leanTree}(h, k)$ computes an AVL tree of height $h$ that is as lean as possible.
All key in the tree will be integers that are bigger than $k$.  The definition by induction:
  - $\texttt{leanTree}(0, k) = \texttt{Nil}$,

    because there is only one AVL tree of height $0$ and this is the tree $\texttt{Nil}$.
  - $\texttt{leanTree}(1, k) = \texttt{Node}(k+1,0,\texttt{Nil}, \texttt{Nil})$,

    since, if we abstract from the actual keys and values, there is exactly one AVL tree of height $1$.
  - $\texttt{leanTree}(h+1,k).\texttt{maxKey}() = l \rightarrow 
     \texttt{leanTree}(h+2,k) = \texttt{Node}\bigl(l+1,\,0,\,\texttt{leanTree}(h+1,k),\,\texttt{leanTree}(h, l+1)\bigr)
    $.

In [ ]:
function leanTree(h: number, k: number): AVLTree<number, null> {
    if (h === 0) return new AVLTree<number, null>();
    if (h === 1) return new AVLTree<number, null>(k + 1, null, new AVLTree<number, null>(), new AVLTree<number, null>());
    
    const left = leanTree(h - 1, k);
    const l = left.maxKey();
    
    if (typeof l !== 'number') throw new Error("leanTree: maxKey returned non-number");
    
    return new AVLTree<number, null>(l + 1, null, left, leanTree(h - 2, l + 1));
}

In [ ]:
const l = leanTree(6, 0);
await renderTree(l);

In [ ]:
for (let k = 0; k < 6; k++) {
    const l = leanTree(k, 0);
    console.log(`Height ${k}:`);
    await renderTree(l);
}